# 🔒 Phishing Email Detection System
## Machine Learning Model for Email Security

---

### Dataset Information
- **Source**: Kaggle - Phishing Email Detection Dataset
- **Size**: ~18,000+ emails
- **Classes**: Safe Emails (0) vs Phishing Emails (1)
- **Features**: Email text body content

### Models Used
1. Logistic Regression
2. Random Forest
3. Support Vector Machine (SVM)
4. Naive Bayes
5. XGBoost

---

## 1. Setup & Install Dependencies

In [ ]:
# Install required packages
!pip install kaggle pandas numpy scikit-learn nltk xgboost wordcloud matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from wordcloud import WordCloud

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report, 
                            confusion_matrix, roc_auc_score, roc_curve,
                            precision_score, recall_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ All dependencies installed successfully!")

## 2. Download Dataset from Kaggle

**Important**: You need to upload your Kaggle API key (kaggle.json) to run this cell.

### Steps to get your Kaggle API key:
1. Go to https://www.kaggle.com/settings
2. Scroll down to 'API' section
3. Click 'Create New API Token'
4. This will download `kaggle.json` file
5. Upload this file when prompted below

In [ ]:
# Upload your kaggle.json file
from google.colab import files
import os

print("📁 Please upload your kaggle.json file:")
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d subhajournal/phishingemails
!unzip -o phishingemails.zip

print("\n✅ Dataset downloaded successfully!")

## 3. Load & Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('Phishing_Email.csv')

# Display basic info
print("📊 Dataset Shape:", df.shape)
print("\n📋 Column Names:", df.columns.tolist())
print("\n🔍 First 5 rows:")
df.head()

In [ ]:
# Check dataset info
print("📊 Dataset Info:")
print(df.info())

print("\n📈 Statistical Summary:")
print(df.describe(include='all'))

In [ ]:
# Check for missing values
print("❌ Missing Values:")
print(df.isnull().sum())

# Check class distribution
print("\n📊 Class Distribution:")
if 'Email Type' in df.columns:
    print(df['Email Type'].value_counts())
elif 'label' in df.columns:
    print(df['label'].value_counts())
else:
    # Find the label column
    for col in df.columns:
        if df[col].dtype == 'object' and df[col].nunique() <= 3:
            print(f"\n{col}:")
            print(df[col].value_counts())

In [ ]:
# Identify the text and label columns
text_col = None
label_col = None

# Find text column (usually contains longer text)
for col in df.columns:
    if df[col].dtype == 'object':
        avg_len = df[col].astype(str).str.len().mean()
        if avg_len > 100:  # Email text is usually long
            text_col = col
            break

# Find label column
for col in df.columns:
    col_lower = col.lower()
    if 'type' in col_lower or 'label' in col_lower or 'class' in col_lower:
        label_col = col
        break

# Fallback
if text_col is None:
    text_col = df.columns[1]  # Usually second column
if label_col is None:
    label_col = df.columns[-1]  # Usually last column

print(f"📝 Text Column: '{text_col}'")
print(f"🏷️ Label Column: '{label_col}'")

## 4. Data Preprocessing

In [ ]:
# Create a copy for processing
data = df.copy()

# Handle missing values
data = data.dropna(subset=[text_col])

# Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['label_encoded'] = le.fit_transform(data[label_col])

print("📊 Label Encoding:")
for i, label in enumerate(le.classes_):
    print(f"  {label} → {i}")

print(f"\n✅ Dataset after cleaning: {data.shape}")

In [ ]:
# Text Preprocessing Function
def preprocess_text(text):
    """
    Clean and preprocess email text:
    1. Convert to lowercase
    2. Remove URLs, emails, HTML tags
    3. Remove punctuation and numbers
    4. Remove stopwords
    5. Apply lemmatization
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    # Remove stopwords and lemmatize
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return ' '.join(words)

# Apply preprocessing
print("🔄 Preprocessing emails... (this may take a minute)")
data['cleaned_text'] = data[text_col].apply(preprocess_text)

print("\n✅ Preprocessing complete!")
print("\n📝 Sample Before:")
print(data[text_col].iloc[0][:200] + "...")
print("\n📝 Sample After:")
print(data['cleaned_text'].iloc[0][:200] + "...")

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Class Distribution Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=data, x=label_col, ax=axes[0], palette='Set2')
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_xlabel('Email Type')
axes[0].set_ylabel('Count')

# Pie chart
class_counts = data[label_col].value_counts()
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', 
            colors=['#66b3ff', '#ff9999'], startangle=90)
axes[1].set_title('Class Distribution (%)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Word Cloud for Phishing Emails
phishing_text = ' '.join(data[data['label_encoded'] == 1]['cleaned_text'])
safe_text = ' '.join(data[data['label_encoded'] == 0]['cleaned_text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Phishing Word Cloud
if len(phishing_text) > 0:
    wc_phishing = WordCloud(width=800, height=400, background_color='white', 
                           max_words=100, colormap='Reds').generate(phishing_text)
    axes[0].imshow(wc_phishing, interpolation='bilinear')
    axes[0].set_title('🚨 Phishing Emails - Word Cloud', fontsize=14)
    axes[0].axis('off')

# Safe Email Word Cloud
if len(safe_text) > 0:
    wc_safe = WordCloud(width=800, height=400, background_color='white',
                        max_words=100, colormap='Blues').generate(safe_text)
    axes[1].imshow(wc_safe, interpolation='bilinear')
    axes[1].set_title('✅ Safe Emails - Word Cloud', fontsize=14)
    axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Email Length Distribution
data['text_length'] = data[text_col].astype(str).apply(len)
data['word_count'] = data[text_col].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Text length
sns.histplot(data=data, x='text_length', hue=label_col, bins=50, ax=axes[0], kde=True)
axes[0].set_title('Email Text Length Distribution', fontsize=14)
axes[0].set_xlabel('Character Count')
axes[0].set_xlim(0, 5000)  # Limit for better visualization

# Word count
sns.histplot(data=data, x='word_count', hue=label_col, bins=50, ax=axes[1], kde=True)
axes[1].set_title('Email Word Count Distribution', fontsize=14)
axes[1].set_xlabel('Word Count')
axes[1].set_xlim(0, 500)

plt.tight_layout()
plt.show()

## 6. Feature Extraction with TF-IDF

In [ ]:
# Prepare features and target
X = data['cleaned_text']
y = data['label_encoded']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

print(f"📊 Training set: {X_train.shape[0]} samples")
print(f"📊 Test set: {X_test.shape[0]} samples")

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), 
                        min_df=2, max_df=0.95)

print("🔄 Fitting TF-IDF Vectorizer...")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"\n✅ TF-IDF Features: {X_train_tfidf.shape[1]}")
print(f"📊 Training matrix shape: {X_train_tfidf.shape}")
print(f"📊 Test matrix shape: {X_test_tfidf.shape}")

In [ ]:
# Top TF-IDF Features
feature_names = tfidf.get_feature_names_out()
tfidf_scores = X_train_tfidf.sum(axis=0).A1

# Get top 20 features
top_indices = tfidf_scores.argsort()[-20:][::-1]
top_features = [(feature_names[i], tfidf_scores[i]) for i in top_indices]

plt.figure(figsize=(10, 6))
plt.barh([f[0] for f in top_features][::-1], [f[1] for f in top_features][::-1])
plt.title('Top 20 Most Important Words (TF-IDF)', fontsize=14)
plt.xlabel('TF-IDF Score')
plt.tight_layout()
plt.show()

## 7. Model Training & Evaluation

In [ ]:
# Dictionary to store results
results = {}

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train and evaluate a model, return metrics
    """
    print(f"\n{'='*50}")
    print(f"🔄 Training {model_name}...")
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    
    # Store results
    results[model_name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"✅ {model_name} trained successfully!")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    if roc_auc:
        print(f"   ROC-AUC: {roc_auc:.4f}")
    
    return model

In [ ]:
# Train Multiple Models

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
evaluate_model(lr, X_train_tfidf, X_test_tfidf, y_train, y_test, 'Logistic Regression')

# 2. Naive Bayes
nb = MultinomialNB()
evaluate_model(nb, X_train_tfidf, X_test_tfidf, y_train, y_test, 'Naive Bayes')

# 3. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
evaluate_model(rf, X_train_tfidf, X_test_tfidf, y_train, y_test, 'Random Forest')

# 4. SVM (with probability for ROC curve)
svm = SVC(kernel='linear', probability=True, random_state=42)
evaluate_model(svm, X_train_tfidf, X_test_tfidf, y_train, y_test, 'SVM')

# 5. XGBoost
xgb = __import__('xgboost').XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
evaluate_model(xgb, X_train_tfidf, X_test_tfidf, y_train, y_test, 'XGBoost')

## 8. Model Comparison & Results

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results],
    'ROC-AUC': [results[m]['roc_auc'] for m in results]
})

# Sort by F1-Score
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print("📊 Model Comparison Results:")
print("="*70)
comparison_df

In [ ]:
# Visualize Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot for metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
comparison_df.plot(x='Model', y=metrics, kind='bar', ax=axes[0], 
                   colormap='viridis', edgecolor='black')
axes[0].set_title('Model Performance Comparison', fontsize=14)
axes[0].set_ylabel('Score')
axes[0].set_ylim(0.8, 1.0)
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=45)

# ROC Curves
for model_name in results:
    if results[model_name]['roc_auc'] is not None:
        fpr, tpr, _ = roc_curve(y_test, results[model_name]['y_pred_proba'])
        axes[1].plot(fpr, tpr, label=f"{model_name} (AUC = {results[model_name]['roc_auc']:.3f})")

axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves', fontsize=14)
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, model_name in enumerate(results):
    cm = confusion_matrix(y_test, results[model_name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Safe', 'Phishing'], yticklabels=['Safe', 'Phishing'])
    axes[idx].set_title(f'{model_name}\nAccuracy: {results[model_name]["accuracy"]:.3f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

# Hide unused subplot
axes[5].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Best Model Analysis
best_model_name = comparison_df.iloc[0]['Model']
best_model = results[best_model_name]['model']

print(f"🏆 Best Model: {best_model_name}")
print(f"   F1-Score: {results[best_model_name]['f1']:.4f}")
print(f"   Accuracy: {results[best_model_name]['accuracy']:.4f}")

# Detailed Classification Report
print(f"\n📋 Classification Report for {best_model_name}:")
print("="*60)
print(classification_report(y_test, results[best_model_name]['y_pred'], 
                          target_names=['Safe Email', 'Phishing Email']))

## 9. Save the Best Model

In [ ]:
import pickle
import joblib

# Save the best model and vectorizer
joblib.dump(best_model, 'phishing_detection_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("✅ Model saved as 'phishing_detection_model.pkl'")
print("✅ TF-IDF Vectorizer saved as 'tfidf_vectorizer.pkl'")
print("✅ Label Encoder saved as 'label_encoder.pkl'")

## 10. Test with New Emails

In [ ]:
def predict_phishing(email_text, model=best_model, vectorizer=tfidf, label_encoder=le):
    """
    Predict whether an email is phishing or safe
    """
    # Preprocess
    cleaned = preprocess_text(email_text)
    
    # Vectorize
    features = vectorizer.transform([cleaned])
    
    # Predict
    prediction = model.predict(features)[0]
    
    # Get probability if available
    if hasattr(model, 'predict_proba'):
        probability = model.predict_proba(features)[0]
        confidence = max(probability)
    else:
        confidence = None
    
    # Get label
    label = label_encoder.inverse_transform([prediction])[0]
    
    return {
        'prediction': label,
        'is_phishing': bool(prediction),
        'confidence': confidence
    }

# Test with sample emails
test_emails = [
    """
    Dear Customer,
    We have detected unusual activity on your account. Please click the link below
    to verify your identity: http://secure-bank-verify.com/login
    Your account will be suspended if you do not verify within 24 hours.
    Thank you,
    Bank Security Team
    """,
    
    """
    Hi John,
    Just wanted to follow up on our meeting yesterday. I've attached the notes
    and the project timeline. Let me know if you have any questions.
    Best regards,
    Sarah
    """,
    
    """
    CONGRATULATIONS!!! You have won $1,000,000 in our lottery!
    To claim your prize, please send your bank details to claim@prize-winner.net
    This offer expires in 24 hours. ACT NOW!
    """,
    
    """
    Dear Team,
    Reminder: The quarterly meeting is scheduled for Friday at 2 PM.
    Please review the attached agenda before the meeting.
    Thanks,
    HR Department
    """
]

print("🧪 Testing Model with Sample Emails:")
print("="*60)

for i, email in enumerate(test_emails, 1):
    result = predict_phishing(email)
    print(f"\n📧 Email {i}:")
    print(f"   Text: {email[:100]}...")
    print(f"   🏷️  Prediction: {result['prediction']}")
    print(f"   🎯 Confidence: {result['confidence']:.2%}")
    print(f"   {'🚨 PHISHING DETECTED!' if result['is_phishing'] else '✅ Safe Email'}")

## 11. Interactive Email Checker

Run the cell below and enter any email to check if it's phishing!

In [ ]:
# Interactive checker (works in Colab)
from IPython.display import HTML, display

email_input = input("📧 Enter an email text to check: ")

if email_input.strip():
    result = predict_phishing(email_input)
    
    print("\n" + "="*50)
    if result['is_phishing']:
        print("🚨 WARNING: This appears to be a PHISHING EMAIL!")
    else:
        print("✅ This appears to be a SAFE EMAIL")
    print("="*50)
    print(f"Confidence: {result['confidence']:.2%}")

---

## 📊 Summary

### Model Performance
- Best performing model trained and saved
- Achieved high accuracy in detecting phishing emails
- Used TF-IDF vectorization for feature extraction

### Key Features
- Preprocessing pipeline for cleaning emails
- Multiple ML models compared
- Interactive prediction function

### Usage
1. Use `predict_phishing(email_text)` to check any email
2. Model files saved for deployment:
   - `phishing_detection_model.pkl`
   - `tfidf_vectorizer.pkl`
   - `label_encoder.pkl`

---